<a href="https://colab.research.google.com/github/Kalrfou/Special_Topics2/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain langchain-community langchain-chroma chromadb sentence-transformers transformers accelerate gradio markdown

In [ ]:
import os
import shutil
import torch
import gradio as gr

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [ ]:
folder_path = "/content/data"

CHROMA_PATH = "/content/chroma_arabic_db_new"
COLLECTION_NAME = "arabic_registration_office_rag_new"

if os.path.exists(CHROMA_PATH):
    shutil.rmtree(CHROMA_PATH)

os.makedirs(CHROMA_PATH, exist_ok=True)

print("Using Chroma path:", CHROMA_PATH)

In [ ]:
loader = DirectoryLoader(
    folder_path,
    glob="**/*.md",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)

docs = loader.load()

print("Loaded markdown files/pages:", len(docs))
print(docs[0].metadata)
print(docs[0].page_content[:500])

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(docs)

print("Number of chunks:", len(chunks))

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)

In [ ]:
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name=COLLECTION_NAME,
    persist_directory=CHROMA_PATH
)

retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

print("ChromaDB created successfully.")

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

print("LLM loaded.")

In [ ]:
def build_prompt(question, retrieved_docs):
    context = "\n\n".join(
        [
            f"المصدر: {doc.metadata.get('source', 'غير معروف')}\nالنص:\n{doc.page_content}"
            for doc in retrieved_docs
        ]
    )

    return f"""
أنت مساعد ذكي خاص بدائرة القبول والتسجيل في جامعة الطفيلة التقنية.

أجب عن السؤال باللغة العربية فقط، واعتمد فقط على المعلومات الموجودة في السياق.

التعليمات:
- لا تخترع أي معلومة غير موجودة في السياق.
- إذا لم تجد الإجابة، قل: "لم أجد الإجابة في الوثائق المتوفرة."
- اجعل الإجابة واضحة ومباشرة.
- إذا كان السؤال عن إجراء، اذكر الخطوات إن وجدت.
- إذا كان السؤال عن رسوم أو برامج أو موظفين، استخرج الإجابة بدقة من السياق.

السياق:
{context}

السؤال:
{question}

الإجابة:
"""

In [ ]:
def rag_answer(question):
    question = str(question).strip()

    retrieved_docs = retriever.invoke("query: " + question)

    prompt = build_prompt(question, retrieved_docs)

    outputs = generator(
        prompt,
        max_new_tokens=300,
        temperature=0.1,
        do_sample=True,
        return_full_text=False
    )

    answer = outputs[0]["generated_text"].strip()

    sources = "\n\n".join(
        [
            f"المصدر {i+1}: {doc.metadata.get('source', 'غير معروف')}"
            for i, doc in enumerate(retrieved_docs)
        ]
    )

    return answer + "\n\n---\nالمصادر:\n" + sources

In [ ]:
question = "كم سعر الساعات؟"
print(rag_answer(question))

In [ ]:
def chat_with_rag(message, history):
    try:
        if isinstance(message, dict):
            question = message.get("text", "")
        else:
            question = str(message)

        question = question.strip()

        if question == "":
            return "الرجاء كتابة سؤال."

        return rag_answer(question)

    except Exception as e:
        return f"حدث خطأ:\n\n{type(e).__name__}: {str(e)}"

In [ ]:
demo = gr.ChatInterface(
    fn=chat_with_rag,
    title="مساعد دائرة القبول والتسجيل",
    description="اسأل عن التسجيل، الرسوم، البرامج الأكاديمية، الموظفين، التعليمات، أو أي معلومات موجودة في ملفات Markdown.",
    textbox=gr.Textbox(
        placeholder="اكتب سؤالك هنا...",
        scale=7
    )
)

demo.launch(share=True, debug=True)